In [1]:
import pandas as pd 
import numpy as np

In [ ]:
xl = pd.ExcelFile("Data/Matthieu_Soumaya_Dec2025.xlsx")

# Afficher les noms des feuilles
# print(xl.sheet_names)
# ['Feuille1', 'Feuille2', 'Feuille3']
print(f"Nombre de feuille : ",{len(xl.sheet_names)})
# une par ligne
for nom in xl.sheet_names:
    print(nom)

## Feuille infos statiques 

In [ ]:
df_00 = pd.read_excel("Data/Matthieu_Soumaya_Dec2025.xlsx", sheet_name="AGE_SEX_DUREE")
print(df_00.shape)
df_00.head()


In [ ]:
df_01 = pd.read_excel("Data/Matthieu_Soumaya_Dec2025.xlsx", sheet_name=" Mutation_GBA_LRRK2")
display(df_01.head())
print(df_01.shape)

In [ ]:
df_00_01 = pd.merge(df_00, df_01, on="SUBJID", how="left")

In [ ]:
df_00_01

In [ ]:
# ══════════════════════════════════════════════════════════
# Afficher les colonnes dupliquées : type + % NaN
# ══════════════════════════════════════════════════════════
print("=" * 65)
print(f"{'Colonne':<20} {'Type _x':<12} {'Type _y':<12} {'NaN _x':>8} {'NaN _y':>8}")
print("=" * 65)

for col_x in cols_x:
    col_y   = col_x.replace("_x", "_y")
    col_nom = col_x.replace("_x", "")
    nan_x   = df_00_01[col_x].isna().mean() * 100
    nan_y   = df_00_01[col_y].isna().mean() * 100
    print(f"{col_nom:<20} {str(df_00_01[col_x].dtype):<12} {str(df_00_01[col_y].dtype):<12} {nan_x:>7.1f}% {nan_y:>7.1f}%")


In [ ]:
# ══════════════════════════════════════════════════════════
#Harmoniser les types (_y → même type que _x)
# ══════════════════════════════════════════════════════════
for col_x in cols_x:
    col_y = col_x.replace("_x", "_y")
    try:
        df_00_01[col_y] = df_00_01[col_y].astype(df_00_01[col_x].dtype)
    except (ValueError, TypeError):
        # Si la conversion échoue, on force en string pour pouvoir comparer
        df_00_01[col_x] = df_00_01[col_x].astype(str)
        df_00_01[col_y] = df_00_01[col_y].astype(str)

In [ ]:
df_00_01

In [ ]:
# # ══════════════════════════════════════════════════════════
# # Comparer les valeurs et afficher la similarité
# # ══════════════════════════════════════════════════════════
# print("\n" + "=" * 50)
# print(f"{'Colonne':<20} {'Similarité':>12} {'Statut'}")
# print("=" * 50)

# resultats = {}
# for col_x in cols_x:
#     col_y   = col_x.replace("_x", "_y")
#     col_nom = col_x.replace("_x", "")

#     # Comparer uniquement les lignes où les deux sont non-NaN
#     masque  = df_00_01[col_x].notna() & df_00_01[col_y].notna()
#     n_total = masque.sum()

#     if n_total == 0:
#         similarite = 0.0
#     else:
#         n_egaux    = (df_00_01.loc[masque, col_x] == df_00_01.loc[masque, col_y]).sum()
#         similarite = (n_egaux / n_total) * 100

#     statut = "identiques" if similarite == 100 else f"  {100-similarite:.1f}% différences"
#     print(f"{col_nom:<20} {similarite:>11.1f}% {statut}")
#     resultats[col_nom] = similarite

In [ ]:
for col_x in cols_x:
    col_y   = col_x.replace("_x", "_y")
    col_nom = col_x.replace("_x", "")

    # Lignes où _x est NaN mais _y est rempli
    masque = df_00_01[col_x].isna() & df_00_01[col_y].notna()
    lignes = df_00_01[masque][["SUBJID", col_x, col_y]]

    if len(lignes) > 0:
        print(f"\n{'='*50}")
        print(f"{col_nom} — {len(lignes)} ligne(s) à combler")
        print(lignes.to_string(index=False))

In [ ]:
# ══════════════════════════════════════════════════════════
#  Fusionner et combler les NaN
# ══════════════════════════════════════════════════════════
for col_x in cols_x:
    col_y   = col_x.replace("_x", "_y")
    col_nom = col_x.replace("_x", "")
    # Colonnes identiques → on garde _x et on comble ses NaN avec _y
    df_00_01[col_nom] = df_00_01[col_x].fillna(df_00_01[col_y])
    df_00_01.drop(columns=[col_x, col_y], inplace=True)
    print(f"{col_nom} : fusionné (NaN comblés depuis _y)")

In [ ]:

# supprimer la premiére ligne 
df_00_01 = df_00_01.iloc[1:].reset_index(drop=True)


# # Ignorer directement à la lecture
# df = pd.read_excel("fichier.xlsx", skiprows=1)

In [ ]:
df_00_01

In [ ]:
df_00_01.to_excel("Output/info_statiques.xlsx",index=False)

## Feuille Bilan

### Séparer les Feuilles 

#### Poids_Taille_V1_3_5

In [ ]:
df_v1_2_3 = pd.read_excel("Data/Matthieu_Soumaya_Dec2025.xlsx", sheet_name="Poids_Taille_V1_3_5")
print(df_v1_2_3.shape)
df_v1_2_3.head()

In [ ]:
df_v1_2_3["TITRE"].value_counts()

In [ ]:
# Séparer le dataset par valeur de la colonne TITRE
df_v1_Poids_Taille = df_v1_2_3[df_v1_2_3["TITRE"] == "Visite Bilan à 1 an - V1"].reset_index(drop=True)
df_v3_Poids_Taille = df_v1_2_3[df_v1_2_3["TITRE"] == "Visite Bilan à 3 ans - V3"].reset_index(drop=True)
df_v5_Poids_Taille = df_v1_2_3[df_v1_2_3["TITRE"] == "Visite Bilan à 5 ans - V5"].reset_index(drop=True)

In [ ]:
df_v1

#### UPDRS_IV

In [ ]:
df_UPDRS_IV = pd.read_excel("Data/Matthieu_Soumaya_Dec2025.xlsx", sheet_name="UPDRS_IV")
print(df_UPDRS_IV.shape)
df_UPDRS_IV.head()

In [ ]:
df_UPDRS_IV["TITRE"].value_counts()

In [ ]:
# Séparer le dataset par valeur de la colonne TITRE
df_V0_UPDRS = df_UPDRS_IV[df_UPDRS_IV["TITRE"] == "Visite de screening"].reset_index(drop=True)
df_v1_UPDRS = df_UPDRS_IV[df_UPDRS_IV["TITRE"] == "Visite Bilan à 1 an - V1"].reset_index(drop=True)
df_v3_UPDRS = df_UPDRS_IV[df_UPDRS_IV["TITRE"] == "Visite Bilan à 3 ans - V3"].reset_index(drop=True)
df_v5_UPDRS = df_UPDRS_IV[df_UPDRS_IV["TITRE"] == "Visite Bilan à 5 ans - V5"].reset_index(drop=True)

#### UPDRSIII_COMPLET_V0_V1

In [ ]:
df_UPDRSIII_COMPLET_V0_V1 = pd.read_excel("Data/Matthieu_Soumaya_Dec2025.xlsx", sheet_name="UPDRSIII_COMPLET_V0_V1")
print(df_UPDRSIII_COMPLET_V0_V1.shape)
df_UPDRSIII_COMPLET_V0_V1.head()



#### UPDRSIII_COMPLET_V3_V5

In [ ]:
df_UPDRSIII_COMPLET_V3_V5 = pd.read_excel("Data/Matthieu_Soumaya_Dec2025.xlsx", sheet_name="UPDRSIII_COMPLET_V3_V5 ")
print(df_UPDRSIII_COMPLET_V3_V5.shape)
df_UPDRSIII_COMPLET_V3_V5.head()

In [ ]:
df_UPDRSIII_COMPLET_V3_V5["VISIT"].value_counts()

In [ ]:
# Séparer le dataset par valeur de la colonne VISIT
df_v3_UPDRSIII_COMPLET = df_UPDRSIII_COMPLET_V3_V5[df_UPDRSIII_COMPLET_V3_V5["VISIT"] == "Visite Bilan à 3 ans - V3"].reset_index(drop=True)
df_v5_UPDRSIII_COMPLET = df_UPDRSIII_COMPLET_V3_V5[df_UPDRSIII_COMPLET_V3_V5["VISIT"] == "Visite Bilan à 5 ans - V5"].reset_index(drop=True)

#### UPDRSIII_TOTAUX

#### ECMP

In [ ]:
df_ECMP = pd.read_excel("Data/Matthieu_Soumaya_Dec2025.xlsx", sheet_name="ECMP")
print(df_ECMP.shape)
df_ECMP.head()

In [ ]:
df_ECMP["TITRE"].value_counts()

In [ ]:
# Séparer le dataset par valeur de la colonne TITRE
df_V0_ECMP = df_ECMP[df_ECMP["TITRE"] == "Visite de screening"].reset_index(drop=True)
df_v1_ECMP = df_ECMP[df_ECMP["TITRE"] == "Visite Bilan à 1 an - V1"].reset_index(drop=True)
df_v3_ECMP = df_ECMP[df_ECMP["TITRE"] == "Visite Bilan à 3 ans - V3"].reset_index(drop=True)
df_v5_ECMP = df_ECMP[df_ECMP["TITRE"] == "Visite Bilan à 5 ans - V5"].reset_index(drop=True)

#### QUIP

In [ ]:
df_QUIP = pd.read_excel("Data/Matthieu_Soumaya_Dec2025.xlsx", sheet_name="QUIP ")
print(df_QUIP.shape)
df_QUIP.head()

In [ ]:
df_QUIP["TITRE"].value_counts()

In [ ]:
# Séparer le dataset par valeur de la colonne TITRE
df_V0_QUIP = df_QUIP[df_QUIP["TITRE"] == "Visite de screening"].reset_index(drop=True)
df_v1_QUIP = df_QUIP[df_QUIP["TITRE"] == "Visite Bilan à 1 an - V1"].reset_index(drop=True)
df_v3_QUIP = df_QUIP[df_QUIP["TITRE"] == "Visite Bilan à 3 ans - V3"].reset_index(drop=True)
df_v5_QUIP = df_QUIP[df_QUIP["TITRE"] == "Visite Bilan à 5 ans - V5"].reset_index(drop=True)

#### UPPS

In [ ]:
df_UPPS = pd.read_excel("Data/Matthieu_Soumaya_Dec2025.xlsx", sheet_name="UPPS")
print(df_UPPS.shape)
df_UPPS.head()

In [ ]:
df_UPPS["TITRE"].value_counts()

In [ ]:
# Séparer le dataset par valeur de la colonne TITRE
df_V0_QUIP = df_QUIP[df_QUIP["TITRE"] == "Visite de screening"].reset_index(drop=True)
df_v1_QUIP = df_QUIP[df_QUIP["TITRE"] == "Visite Bilan à 1 an - V1"].reset_index(drop=True)
df_v3_QUIP = df_QUIP[df_QUIP["TITRE"] == "Visite Bilan à 3 ans - V3"].reset_index(drop=True)
df_v5_QUIP = df_QUIP[df_QUIP["TITRE"] == "Visite Bilan à 5 ans - V5"].reset_index(drop=True)

#### MOCA_V0_V3

In [ ]:
df_MOCA_V0_V3 = pd.read_excel("Data/Matthieu_Soumaya_Dec2025.xlsx", sheet_name="MOCA_V0_V3")
print(df_MOCA_V0_V3.shape)
df_MOCA_V0_V3.head()

In [ ]:
df_MOCA_V0_V3["TITRE"].value_counts()

In [ ]:
# Séparer le dataset par valeur de la colonne TITRE
df_V0_MOCA = df_MOCA_V0_V3[df_MOCA_V0_V3["TITRE"] == "Visite de screening"].reset_index(drop=True)
df_v3_MOCA = df_MOCA_V0_V3[df_MOCA_V0_V3["TITRE"] == "Visite Bilan à 3 ans - V3"].reset_index(drop=True)


#### MOCA_V1_V5

In [ ]:
df_MOCA_V1_V5 = pd.read_excel("Data/Matthieu_Soumaya_Dec2025.xlsx", sheet_name="MOCA_V1_V5")
print(df_MOCA_V1_V5.shape)
df_MOCA_V1_V5.head()

In [ ]:
df_MOCA_V1_V5["TITRE"].value_counts()

In [ ]:
# Séparer le dataset par valeur de la colonne TITRE
df_v1_MOCA = df_MOCA_V1_V5[df_MOCA_V1_V5["TITRE"] == "Visite Bilan à 1 an - V1"].reset_index(drop=True)
df_v5_MOCA = df_MOCA_V1_V5[df_MOCA_V1_V5["TITRE"] == "Visite Bilan à 5 ans - V5"].reset_index(drop=True)

#### HAMD

In [ ]:
df_HAMD = pd.read_excel("Data/Matthieu_Soumaya_Dec2025.xlsx", sheet_name="HAMD")
print(df_HAMD.shape)
df_HAMD.head()

In [ ]:
df_HAMD["TITRE"].value_counts()

In [ ]:
# Séparer le dataset par valeur de la colonne TITRE
df_V0_HAMD = df_HAMD[df_HAMD["TITRE"] == "Visite de screening"].reset_index(drop=True)
df_v1_HAMD = df_HAMD[df_HAMD["TITRE"] == "Visite Bilan à 1 an - V1"].reset_index(drop=True)
df_v3_HAMD = df_HAMD[df_HAMD["TITRE"] == "Visite Bilan à 3 ans - V3"].reset_index(drop=True)
df_v5_HAMD = df_HAMD[df_HAMD["TITRE"] == "Visite Bilan à 5 ans - V5"].reset_index(drop=True)

#### HAMA 

In [ ]:
df_HAMA = pd.read_excel("Data/Matthieu_Soumaya_Dec2025.xlsx", sheet_name="HAMA ")
print(df_HAMA.shape)
df_HAMA.head()

In [ ]:
df_HAMA["TITRE"].value_counts()

In [ ]:
# Séparer le dataset par valeur de la colonne TITRE
df_V0_HAMA = df_HAMA[df_HAMA["TITRE"] == "Visite de screening"].reset_index(drop=True)
df_v1_HAMA = df_HAMA[df_HAMA["TITRE"] == "Visite Bilan à 1 an - V1"].reset_index(drop=True)
df_v3_HAMA = df_HAMA[df_HAMA["TITRE"] == "Visite Bilan à 3 ans - V3"].reset_index(drop=True)
df_v5_HAMA = df_HAMA[df_HAMA["TITRE"] == "Visite Bilan à 5 ans - V5"].reset_index(drop=True)

#### LARS

In [ ]:
df_LARS = pd.read_excel("Data/Matthieu_Soumaya_Dec2025.xlsx", sheet_name="LARS ")
print(df_LARS.shape)
df_LARS.head()

In [ ]:
df_LARS["TITRE"].value_counts()

In [ ]:
# Séparer le dataset par valeur de la colonne TITRE
df_V0_LARS = df_LARS[df_LARS["TITRE"] == "Visite de screening"].reset_index(drop=True)
df_v1_LARS = df_LARS[df_LARS["TITRE"] == "Visite Bilan à 1 an - V1"].reset_index(drop=True)
df_v3_LARS = df_LARS[df_LARS["TITRE"] == "Visite Bilan à 3 ans - V3"].reset_index(drop=True)
df_v5_LARS = df_LARS[df_LARS["TITRE"] == "Visite Bilan à 5 ans - V5"].reset_index(drop=True)

#### PDQ39

In [ ]:
df_PDQ39 = pd.read_excel("Data/Matthieu_Soumaya_Dec2025.xlsx", sheet_name="PDQ39")
print(df_PDQ39.shape)
df_PDQ39.head()

In [ ]:
df_PDQ39["VISIT"].value_counts()

In [ ]:
# Séparer le dataset par valeur de la colonne VISIT
df_V0_PDQ39 = df_PDQ39[df_PDQ39["VISIT"] == "Visite de screening"].reset_index(drop=True)
df_v1_PDQ39 = df_PDQ39[df_PDQ39["VISIT"] == "Visite Bilan à 1 an - V1"].reset_index(drop=True)
df_v3_PDQ39 = df_PDQ39[df_PDQ39["VISIT"] == "Visite Bilan à 3 ans - V3"].reset_index(drop=True)
df_v5_PDQ39 = df_PDQ39[df_PDQ39["VISIT"] == "Visite Bilan à 5 ans - V5"].reset_index(drop=True)

#### DIGITSMT_TRAILMT_DKEFS

In [ ]:
df_DIGITSMT_TRAILMT_DKEFS = pd.read_excel("Data/Matthieu_Soumaya_Dec2025.xlsx", sheet_name="DIGITSMT_TRAILMT_DKEFS")
print(df_DIGITSMT_TRAILMT_DKEFS.shape)
df_DIGITSMT_TRAILMT_DKEFS.head()

In [ ]:
df_DIGITSMT_TRAILMT_DKEFS["TITRE"].value_counts()

In [ ]:
# Séparer le dataset par valeur de la colonne VISIT
df_V0_DIGITSMT_TRAILMT_DKEFS = df_DIGITSMT_TRAILMT_DKEFS[df_DIGITSMT_TRAILMT_DKEFS["TITRE"] == "Visite de screening"].reset_index(drop=True)
df_v1_DIGITSMT_TRAILMT_DKEFS = df_DIGITSMT_TRAILMT_DKEFS[df_DIGITSMT_TRAILMT_DKEFS["TITRE"] == "Visite Bilan à 1 an - V1"].reset_index(drop=True)
df_v3_DIGITSMT_TRAILMT_DKEFS = df_DIGITSMT_TRAILMT_DKEFS[df_DIGITSMT_TRAILMT_DKEFS["TITRE"] == "Visite Bilan à 3 ans - V3"].reset_index(drop=True)
df_v5_DIGITSMT_TRAILMT_DKEFS = df_DIGITSMT_TRAILMT_DKEFS[df_DIGITSMT_TRAILMT_DKEFS["TITRE"] == "Visite Bilan à 5 ans - V5"].reset_index(drop=True)

#### CONSO_SPECIFIQUE 

In [ ]:
df_CONSO_SPECIFIQUE = pd.read_excel("Data/Matthieu_Soumaya_Dec2025.xlsx", sheet_name="CONSO_SPECIFIQUE ")
print(df_CONSO_SPECIFIQUE.shape)
df_CONSO_SPECIFIQUE.head()

In [ ]:
df_CONSO_SPECIFIQUE["NUM"].value_counts()

In [ ]:
# Séparer le dataset par valeur de la colonne VISIT
df_V0_CONSO_SPECIFIQUE = df_CONSO_SPECIFIQUE[df_CONSO_SPECIFIQUE["NUM"] == "1"].reset_index(drop=True)
df_vc_CONSO_SPECIFIQUE = df_CONSO_SPECIFIQUE[df_CONSO_SPECIFIQUE["NUM"] == "2"].reset_index(drop=True)
df_v1_CONSO_SPECIFIQUE = df_CONSO_SPECIFIQUE[df_CONSO_SPECIFIQUE["NUM"] == "3"].reset_index(drop=True)
df_v3_CONSO_SPECIFIQUE = df_CONSO_SPECIFIQUE[df_CONSO_SPECIFIQUE["NUM"] == "4"].reset_index(drop=True)
df_v5_CONSO_SPECIFIQUE = df_CONSO_SPECIFIQUE[df_CONSO_SPECIFIQUE["NUM"] == "5"].reset_index(drop=True)

#### Feuil4

In [ ]:
df_Feuil4 = pd.read_excel("Data/Matthieu_Soumaya_Dec2025.xlsx", sheet_name="Feuil4")
print(df_Feuil4.shape)
df_Feuil4.head()

#### LEDD

In [ ]:
df_LEDD = pd.read_excel("Data/Matthieu_Soumaya_Dec2025.xlsx", sheet_name="LEDD")
print(df_LEDD.shape)
df_LEDD.head()

In [ ]:
df_LEDD["Num"].value_counts()

In [ ]:
# Séparer le dataset par valeur de la colonne VISIT
df_V0_LEDD = df_LEDD[df_LEDD["Num"] == 1].reset_index(drop=True)
df_vc_LEDD = df_LEDD[df_LEDD["Num"] == 2].reset_index(drop=True)
df_v1_LEDD = df_LEDD[df_LEDD["Num"] == 3].reset_index(drop=True)
df_v3_LEDD = df_LEDD[df_LEDD["Num"] == 4].reset_index(drop=True)
df_v5_LEDD = df_LEDD[df_LEDD["Num"] == 5].reset_index(drop=True)

#### Feuil1

In [ ]:
df_Feuil1 = pd.read_excel("Data/Matthieu_Soumaya_Dec2025.xlsx", sheet_name="Feuil1")
print(df_Feuil1.shape)
df_Feuil1.head()

In [ ]:
df_Feuil1["Num"].value_counts()

In [ ]:
# Séparer le dataset par valeur de la colonne VISIT
df_V0_Feuil1 = df_Feuil1[df_Feuil1["Num"] == 1].reset_index(drop=True)
df_vc_Feuil1 = df_Feuil1[df_Feuil1["Num"] == 2].reset_index(drop=True)
df_v1_Feuil1 = df_Feuil1[df_Feuil1["Num"] == 3].reset_index(drop=True)
df_v3_Feuil1 = df_Feuil1[df_Feuil1["Num"] == 4].reset_index(drop=True)
df_v5_Feuil1 = df_Feuil1[df_Feuil1["Num"] == 5].reset_index(drop=True)

#### Feuil2

In [ ]:
df_Feuil2 = pd.read_excel("Data/Matthieu_Soumaya_Dec2025.xlsx", sheet_name="Feuil2")
print(df_Feuil2.shape)
df_Feuil2.head()

#### PSYCHOTROPES

In [ ]:
df_PSYCHOTROPES = pd.read_excel("Data/Matthieu_Soumaya_Dec2025.xlsx", sheet_name="PSYCHOTROPES")
print(df_PSYCHOTROPES.shape)
df_PSYCHOTROPES.head()

In [ ]:
df_PSYCHOTROPES["NUM"].value_counts()

In [ ]:
# Séparer le dataset par valeur de la colonne VISIT
df_V0_PSYCHOTROPES = df_PSYCHOTROPES[df_PSYCHOTROPES["NUM"] == "1"].reset_index(drop=True)
df_vc_PSYCHOTROPES = df_PSYCHOTROPES[df_PSYCHOTROPES["NUM"] == "2"].reset_index(drop=True)
df_v1_PSYCHOTROPES = df_PSYCHOTROPES[df_PSYCHOTROPES["NUM"] == "3"].reset_index(drop=True)
df_v3_PSYCHOTROPES = df_PSYCHOTROPES[df_PSYCHOTROPES["NUM"] == "4"].reset_index(drop=True)
df_v5_PSYCHOTROPES = df_PSYCHOTROPES[df_PSYCHOTROPES["NUM"] == "5"].reset_index(drop=True)


#### AUTRE_PARKINSON

In [ ]:
df_AUTRE_PARKINSON = pd.read_excel("Data/Matthieu_Soumaya_Dec2025.xlsx", sheet_name="AUTRE_PARKINSON")
print(df_AUTRE_PARKINSON.shape)
df_AUTRE_PARKINSON.head()

In [ ]:
df_AUTRE_PARKINSON["NUM"].value_counts()

In [ ]:
# Séparer le dataset par valeur de la colonne VISIT
df_V0_AUTRE_PARKINSON = df_AUTRE_PARKINSON[df_AUTRE_PARKINSON["NUM"] == "1"].reset_index(drop=True)
df_vc_AUTRE_PARKINSON = df_AUTRE_PARKINSON[df_AUTRE_PARKINSON["NUM"] == "2"].reset_index(drop=True)
df_v1_AUTRE_PARKINSON = df_AUTRE_PARKINSON[df_AUTRE_PARKINSON["NUM"] == "3"].reset_index(drop=True)
df_v3_AUTRE_PARKINSON = df_AUTRE_PARKINSON[df_AUTRE_PARKINSON["NUM"] == "4"].reset_index(drop=True)
df_v5_AUTRE_PARKINSON = df_AUTRE_PARKINSON[df_AUTRE_PARKINSON["NUM"] == "5"].reset_index(drop=True)


#### FREQUENCE_V1_3_5

In [ ]:
df_FREQUENCE_V1_3_5 = pd.read_excel("Data/Matthieu_Soumaya_Dec2025.xlsx", sheet_name="FREQUENCE_V1_3_5 ")
print(df_FREQUENCE_V1_3_5.shape)
df_FREQUENCE_V1_3_5.head()

In [ ]:
df_FREQUENCE_V1_3_5["VISITE"].value_counts()

In [ ]:
# Séparer le dataset par valeur de la colonne VISITE
df_v1_FREQUENCE = df_FREQUENCE_V1_3_5[df_FREQUENCE_V1_3_5["VISITE"] == "Visite Bilan à 1 an - V1"].reset_index(drop=True)
df_v3_FREQUENCE = df_FREQUENCE_V1_3_5[df_FREQUENCE_V1_3_5["VISITE"] == "Visite Bilan à 3 ans - V3"].reset_index(drop=True)
df_v5_FREQUENCE = df_FREQUENCE_V1_3_5[df_FREQUENCE_V1_3_5["VISITE"] == "Visite Bilan à 5 ans - V5"].reset_index(drop=True)


#### DATES_VISITE

In [ ]:
df_DATES_VISITE = pd.read_excel("Data/Matthieu_Soumaya_Dec2025.xlsx", sheet_name="DATES_VISITE")
print(df_DATES_VISITE.shape)
df_DATES_VISITE.head()

In [ ]:
df_V0_DATES_VISITE = df_DATES_VISITE[["SUBJID","D_SCREEN"]]
df_vc_DATES_VISITE = df_DATES_VISITE[["SUBJID","D_CHIR"]]
df_v1_DATES_VISITE = df_DATES_VISITE[["SUBJID","DATE_V1"]]
df_v3_DATES_VISITE = df_DATES_VISITE[["SUBJID","DATE_V3"]]
df_v5_DATES_VISITE = df_DATES_VISITE[["SUBJID","DATE_V5"]]

#### FIN_ETUDE


In [ ]:
df_FIN_ETUDEQUIP = pd.read_excel("Data/Matthieu_Soumaya_Dec2025.xlsx", sheet_name="FIN_ETUDE")
print(df_FIN_ETUDEQUIP.shape)
df_FIN_ETUDEQUIP.head()

In [ ]:
import pandas as pd

with pd.ExcelWriter("Output/info_statiques.xlsx", mode="a", engine="openpyxl") as writer:
    df_FIN_ETUDEQUIP.to_excel(writer, sheet_name="Sheet2", index=False)

#### Fichier Excel 

In [ ]:
import pandas as pd

lis = [name for name, val in globals().items() 
 if isinstance(val, pd.DataFrame) and name.startswith("df_v5")]
display(lis)
print(len(lis))


In [ ]:
import pandas as pd

# ============================================================
#   DOSSIER DE SORTIE  (change si besoin)
# ============================================================
OUTPUT_DIR = "Output"   


# ============================================================
#   GROUPE 1 : fichier V0.xlsx
# ============================================================
sheets_V0 = {
    "DATES_VISITE"                 : df_V0_DATES_VISITE,
    "df_V0"                        : df_V0,
    "Feuil1"                       : df_V0_Feuil1,
    "LEDD"                         : df_V0_LEDD,
    "PSYCHOTROPES"                 : df_V0_PSYCHOTROPES,
    "AUTRE_PARKINSON"              : df_V0_AUTRE_PARKINSON,
    "CONSO_SPECIFIQUE"             : df_V0_CONSO_SPECIFIQUE,
    "DIGITSMT_TRAILMT_DKEFS"       : df_V0_DIGITSMT_TRAILMT_DKEFS,
    "PDQ39"                        : df_V0_PDQ39,
    "LARS"                         : df_V0_LARS,
    "HAMA"                         : df_V0_HAMA,
    "HAMD"                         : df_V0_HAMD,
    "MOCA"                         : df_V0_MOCA,
    "QUIP"                         : df_V0_QUIP,
    "ECMP"                         : df_V0_ECMP,
    "UPDRS"                        : df_V0_UPDRS,
    
}

# ============================================================
#   GROUPE 2 : fichier v1.xlsx
# ============================================================
sheets_v1 = {
    "DATES_VISITE"                 : df_v1_DATES_VISITE,
    "df_v1"                        : df_v1,
    "Feuil1"                       : df_v1_Feuil1,
    "LEDD"                         : df_v1_LEDD,
    "PSYCHOTROPES"                 : df_v1_PSYCHOTROPES,
    "AUTRE_PARKINSON"              : df_v1_AUTRE_PARKINSON,
    "FREQUENCE"                    : df_v1_FREQUENCE,
    "CONSO_SPECIFIQUE"             : df_v1_CONSO_SPECIFIQUE,
    "DIGITSMT_TRAILMT_DKEFS"       : df_v1_DIGITSMT_TRAILMT_DKEFS,
    "PDQ39"                        : df_v1_PDQ39,
    "LARS"                         : df_v1_LARS,
    "HAMA"                         : df_v1_HAMA,
    "HAMD"                         : df_v1_HAMD,
    "MOCA"                         : df_v1_MOCA,
    "QUIP"                         : df_v1_QUIP,
    "ECMP"                         : df_v1_ECMP,
    "UPDRS"                        : df_v1_UPDRS,
    "Poids_Taille"                 : df_v1_Poids_Taille,
}

# ============================================================
#   GROUPE 3 : fichier vc.xlsx
# ============================================================
sheets_vc = {
    "DATES_VISITE"                 : df_vc_DATES_VISITE,
    "CONSO_SPECIFIQUE"             : df_vc_CONSO_SPECIFIQUE,
    "Feuil1"                       : df_vc_Feuil1,
    "LEDD"                         : df_vc_LEDD,
    "PSYCHOTROPES"                 : df_vc_PSYCHOTROPES,
    "AUTRE_PARKINSON"              : df_vc_AUTRE_PARKINSON,
    "CONSO_SPECIFIQUE"             : df_vc_CONSO_SPECIFIQUE,
}

# ============================================================
#   GROUPE 4 : fichier v3.xlsx
# ============================================================
sheets_v3 = {
    "DATES_VISITE"                 : df_v3_DATES_VISITE,
    "df_v3"                        : df_v3,
    "Feuil1"                       : df_v3_Feuil1,
    "LEDD"                         : df_v3_LEDD,
    "PSYCHOTROPES"                 : df_v3_PSYCHOTROPES,
    "AUTRE_PARKINSON"              : df_v3_AUTRE_PARKINSON,
    "FREQUENCE"                    : df_v3_FREQUENCE,
    "CONSO_SPECIFIQUE"             : df_v3_CONSO_SPECIFIQUE,
    "DIGITSMT_TRAILMT_DKEFS"       : df_v3_DIGITSMT_TRAILMT_DKEFS,
    "PDQ39"                        : df_v3_PDQ39,
    "LARS"                         : df_v3_LARS,
    "HAMA"                         : df_v3_HAMA,
    "HAMD"                         : df_v3_HAMD,
    "MOCA"                         : df_v3_MOCA,
    "QUIP"                         : df_v3_QUIP,
    "ECMP"                         : df_v3_ECMP,
    "UPDRSIII_COMPLET"             : df_v3_UPDRSIII_COMPLET,
    "UPDRS"                        : df_v3_UPDRS,
    "Poids_Taille"                 : df_v3_Poids_Taille,
}

# ============================================================
#   GROUPE 5 : fichier v5.xlsx
# ============================================================
sheets_v5 = {
    "DATES_VISITE"                 : df_v5_DATES_VISITE,
    "df_v5"                        : df_v5,
    "Feuil1"                       : df_v5_Feuil1,
    "LEDD"                         : df_v5_LEDD,
    "PSYCHOTROPES"                 : df_v5_PSYCHOTROPES,
    "AUTRE_PARKINSON"              : df_v5_AUTRE_PARKINSON,
    "FREQUENCE"                    : df_v5_FREQUENCE,
    "CONSO_SPECIFIQUE"             : df_v5_CONSO_SPECIFIQUE,
    "DIGITSMT_TRAILMT_DKEFS"       : df_v5_DIGITSMT_TRAILMT_DKEFS,
    "PDQ39"                        : df_v5_PDQ39,
    "LARS"                         : df_v5_LARS,
    "HAMA"                         : df_v5_HAMA,
    "HAMD"                         : df_v5_HAMD,
    "MOCA"                         : df_v5_MOCA,
    "QUIP"                         : df_v5_QUIP,
    "ECMP"                         : df_v5_ECMP,
    "UPDRSIII_COMPLET"             : df_v5_UPDRSIII_COMPLET,
    "UPDRS"                        : df_v5_UPDRS,
    "Poids_Taille"                 : df_v5_Poids_Taille,
}


# ─── Ne touche pas au code en dessous ───────────────────────

all_groups = {
    "V0" : sheets_V0,
    "v1" : sheets_v1,
    "vc" : sheets_vc,
    "v3" : sheets_v3,
    "v5" : sheets_v5,
}

for version, sheets in all_groups.items():
    filepath = f"{OUTPUT_DIR}/{version}.xlsx"
    with pd.ExcelWriter(filepath, engine="openpyxl") as writer:
        for sheet_name, df in sheets.items():
            df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
    print(f"[OK] {filepath}  →  {len(sheets)} feuilles")

In [ ]:
df

### Bilan V0

In [ ]:
df_V0 = pd.read_excel("Data/Matthieu_Soumaya_Dec2025.xlsx", sheet_name="Poids_Taille V0")
print(df_V0.shape)
df_V0.head()